# Check overfit — อ่านผลที่รันแล้ว

ใช้ `history.csv` ที่มีอยู่เพื่อวาด train/validation loss ไม่โหลดโมเดล ไม่ฝึกใหม่ ไม่ต้องใช้ GPU
เปิด notebook นี้บน Colab แล้วรัน 2 เซลล์ ใช้ CPU runtime ได้ และไม่ต้องอัปโหลด Colab ZIP ใหม่
แก้ `RUN_ROOT` ให้ตรงโฟลเดอร์ run ที่มี `CNN/results/` ของคุณ หากใช้ output อื่น

บันทึกกราฟรวมที่ `comparison/training_curves.png` และกราฟรายวิธีที่
`CNN/results/<condition_id>/training_curves.png` พร้อม `comparison/overfit_summary.csv`
ถ้ารันเสร็จเพียง baseline ก็วาด baseline ได้ วิธีที่ยังไม่มี history จะแจ้งว่าขาด

เส้น **Train (eval mode)** เทียบ **Validation** ใช้ดู gap ได้ตรงกว่า loss ระหว่าง update ที่เปิด dropout
เมื่อ train loss ลดแต่ validation loss เพิ่มหลัง best epoch ต่อเนื่อง เป็นสัญญาณ overfit
เส้นประสีเขียวระบุ epoch ที่ validation loss ต่ำที่สุดใน history ที่มี ไม่ใช่ผล test
หาก history รุ่นเก่าไม่มี train_eval_loss จะใช้ train_loss พร้อมระบุว่าเป็น loss ระหว่างฝึก

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUN_ROOT = Path("/content/drive/MyDrive/Notebook_3_beta_class/runs/class200_e10")
if not IN_COLAB:
    candidates = [Path.cwd(), Path.cwd() / "Notebook_3 beta class 200",
                  Path.cwd() / "Notebook_3 beta class", *Path.cwd().parents]
    local_root = next((p for p in candidates if (p / "runs/class200_e10/CNN/results").is_dir()), Path.cwd())
    RUN_ROOT = local_root / "runs/class200_e10"
print("Read saved histories from:", RUN_ROOT)

In [ ]:
"""Plot saved CNN histories only; no model imports, training, or run locks."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CONDITIONS = ("baseline", "dwt_L1_L4_fusion", "dft_low_mid_high_fusion")


def plot_overfit(run_root, condition_ids=CONDITIONS, display_plots=True):
    run_root = Path(run_root)
    frames, missing = {}, []
    for condition in condition_ids:
        history = run_root / "CNN/results" / condition / "history.csv"
        if not history.is_file():
            missing.append(condition)
            continue
        frame = pd.read_csv(history).sort_values("epoch")
        if frame.empty or not {"epoch", "train_loss", "validation_loss"} <= set(frame):
            raise ValueError(f"Invalid or empty history: {history}")
        if frame.epoch.duplicated().any():
            raise ValueError(f"Duplicate epochs: {history}")
        for name in ("epoch", "train_loss", "validation_loss", "train_eval_loss"):
            if name in frame:
                frame[name] = pd.to_numeric(frame[name], errors="raise")
                if not np.isfinite(frame[name]).all():
                    raise ValueError(f"Nonfinite {name}: {history}")
        frames[condition] = frame
    if not frames:
        raise FileNotFoundError(f"No CNN/results/<condition>/history.csv under {run_root}. Set RUN_ROOT to your saved run.")

    def draw(ax, condition, frame):
        if "train_eval_loss" in frame:
            ax.plot(frame.epoch, frame.train_eval_loss, "o-", label="Train (eval mode)", color="#2563eb")
            ax.plot(frame.epoch, frame.train_loss, "--", alpha=0.6, label="Train (dropout on)", color="#6b7280")
        else:
            ax.plot(frame.epoch, frame.train_loss, "o-", label="Train (during updates)", color="#2563eb")
        ax.plot(frame.epoch, frame.validation_loss, "o-", label="Validation", color="#ea580c")
        best = frame.loc[frame.validation_loss.idxmin(), "epoch"]
        ax.axvline(best, linestyle=":", alpha=0.65, color="#15803d", label=f"Best validation: epoch {best:g}")
        ax.set(title=condition, xlabel="Epoch", ylabel="BCE loss", xticks=frame.epoch)
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)

    combined, axes = plt.subplots(1, len(frames), figsize=(5 * len(frames), 4), squeeze=False)
    paths, summary = {}, []
    for ax, (condition, frame) in zip(axes[0], frames.items()):
        draw(ax, condition, frame)
        fig, single = plt.subplots(figsize=(7, 4.5))
        draw(single, condition, frame)
        fig.tight_layout()
        destination = run_root / "CNN/results" / condition / "training_curves.png"
        fig.savefig(destination, dpi=160, bbox_inches="tight")
        plt.close(fig)
        paths[condition] = destination
        best = frame.loc[frame.validation_loss.idxmin()]
        last = frame.iloc[-1]
        row = {"condition_id": condition, "epochs_recorded": len(frame),
               "best_validation_epoch": int(best.epoch), "best_validation_loss": float(best.validation_loss),
               "last_epoch": int(last.epoch), "last_validation_loss": float(last.validation_loss)}
        if "train_eval_loss" in frame:
            row.update(last_train_eval_loss=float(last.train_eval_loss),
                       last_validation_minus_train_eval_loss=float(last.validation_loss - last.train_eval_loss))
        summary.append(row)
    combined.suptitle("Check overfit: train / validation loss")
    combined.tight_layout()
    output = run_root / "comparison"
    output.mkdir(parents=True, exist_ok=True)
    combined_path = output / "training_curves.png"
    combined.savefig(combined_path, dpi=160, bbox_inches="tight")
    plt.close(combined)
    table = pd.DataFrame(summary)
    table.to_csv(output / "overfit_summary.csv", index=False)
    if missing:
        print("No saved history yet:", ", ".join(missing))
    print("Combined plot:", combined_path)
    for condition, path in paths.items():
        print(condition, "plot:", path)
    if display_plots:
        from IPython.display import display, Image
        display(Image(filename=str(combined_path)))
        display(table)
    return {"combined": combined_path, "per_condition": paths, "summary": table, "missing": missing}

report = plot_overfit(RUN_ROOT)
